## Data preparation

* Check if the missing values are presented in the features.
* If there are missing values:
    * For categorical features, replace them with 'NA'
    * For numerical features, replace with with 0.0

In [43]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [44]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data -O data-week-3.csv 

--2026-10-12 15:14:47--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 272.21K  --.-KB/s    in 0.03s   

2026-10-12 15:14:47 (9.42 MB/s) - ‘data-week-3.csv’ saved [278746/278746]



In [45]:
df = pd.read_csv('data-week-3.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [46]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [47]:
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]


df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0)

### train data

In [48]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [49]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [50]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [51]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [52]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [53]:
train_dicts = df_train[categorical + numerical].to_dict(
    orient='records'
)

val_dicts = df_val[categorical + numerical].to_dict(
    orient='records'
)

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

## Question 1
What is the most frequent observation (mode) for the column industry?

* NA
* technology
* healthcare
* retail

In [54]:
df.industry.mode()[0]

'technology'

## Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.


In [55]:
corr_matrix = round(df_full_train[numerical].corr(),2)

upper = corr_matrix.abs().where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

upper.stack().sort_values(ascending=False)

interaction_count         lead_score                  0.92
number_of_courses_viewed  lead_score                  0.76
                          interaction_count           0.72
annual_income             lead_score                  0.24
                          number_of_courses_viewed    0.16
                          interaction_count           0.13
dtype: float64

## Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [56]:
categorical = [
    'industry',
    'location',
    'lead_source',
    'employment_status'
]

def mutual_info(series):
    return mutual_info_score(series, y_train)

mi_scores = (
    df_train[categorical]
    .apply(mutual_info)
    .round(2)
    .sort_values(ascending=False)
)
mi_scores

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64

## Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [57]:
def calculate_accuracy(df_train, df_val):
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(
        df_train.to_dict(orient='records')
    )

    X_val = dv.transform(
        df_val.to_dict(orient='records')
    )

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    return accuracy_score(y_val, y_pred)

In [58]:
accuracy = calculate_accuracy(df_train, df_val)

print(accuracy)          
print(round(accuracy, 2))

0.651
0.65


## Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [59]:
features_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

differences = {}

for feature in features_to_test:
    df_train_reduced = df_train.drop(columns=[feature])
    df_val_reduced = df_val.drop(columns=[feature])

    reduced_accuracy = calculate_accuracy(
        df_train_reduced,
        df_val_reduced
    )

    difference = accuracy - reduced_accuracy
    differences[feature] = difference

    print(
        feature,
        'accuracy:', reduced_accuracy,
        'difference:', difference
    )

min(differences, key=differences.get)

lead_source accuracy: 0.648 difference: 0.0030000000000000027
number_of_courses_viewed accuracy: 0.645 difference: 0.006000000000000005
interaction_count accuracy: 0.593 difference: 0.05800000000000005


'lead_source'

## Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

> **Note**: If there are multiple options, select the smallest `C`.

In [60]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
scores = {}

for c in c_values:
    model = LogisticRegression(
        solver='liblinear',
        C=c,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)

    scores[c] = round(accuracy, 3)

best_c = max(scores, key=scores.get)
print('Best C:', best_c)

Best C: 0.001


## Submit the results

- Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03
- The numerical options are calculated from the pinned 2026 release. Use the value that matches your calculation; do not choose a merely close value.